In [ ]:
# Libraries
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.linear_model import Lasso, ElasticNet, LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.base import clone
from backend_aipw import *

In [1]:
def dgp1(n, rng):
    p = 5
    X = rng.binomial(1, 0.5, size=(n, p))
    D = rng.binomial(1, 0.5, size=n)

    beta0 = np.array([0.5, 1.0, 0.2, 0.4, 1.5])
    beta1 = beta0 + 1.0  # constant ATE = 1

    Y0 = X @ beta0 + rng.normal(0, 1, size=n)
    Y1 = X @ beta1 + rng.normal(0, 1, size=n)
    Y = D * Y1 + (1 - D) * Y0
    
    return X, D, Y, 2.5

In [51]:
def dgp2(n, rng):
    p = int(0.1 * n)
    s = 5

    X = rng.binomial(1, 0.5, size=(n, p))
    D = rng.binomial(1, 0.5, size=n)

    group_vals = rng.uniform(0, 1, size=s)
    beta = np.resize(group_vals, p)

    Y0 = X @ beta + rng.normal(0, 1, size=n)
    Y1 = Y0 + 1.0

    Y = D * Y1 + (1 - D) * Y0
    return X, D, Y, 1.0

In [52]:
def dgp3(n, rng):
    p = int(1.7 * n)

    X = rng.binomial(1, 0.5, size=(n, p))
    D = rng.binomial(1, 0.5, size=n)
 
    beta = rng.uniform(0, 1, size=p)

    Y0 = X @ beta + rng.normal(0, 1, size=n)
    Y1 = Y0 + 1.0

    Y = D * Y1 + (1 - D) * Y0
    return X, D, Y, 1.0

In [53]:
LASSO_GRID = {"alpha": np.logspace(-3, 0, 8)}

EN_GRID = {
    "alpha": np.logspace(-3, 0, 8),
    "l1_ratio": [0.2, 0.5, 0.8]
}

RF_GRID = {
    "max_depth": [3, 5, None],
    "min_samples_leaf": [5, 10, 20]
}

GB_GRID = {
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [2, 3]
}

In [55]:
def tune_once(dgp_func, learners, n, seed=0):
    rng = np.random.default_rng(seed)
    X, D, Y, _ = dgp_func(n, rng)

    tuned = {}

    for name, model in learners.items():
        if name == "Lasso":
            grid = LASSO_GRID
        elif name == "ElasticNet":
            grid = EN_GRID
        elif name == "RF":
            grid = RF_GRID
        elif name == "GB":
            grid = GB_GRID
        else:
            tuned[name] = clone(model)
            continue

        gs = GridSearchCV(
            clone(model),
            grid,
            cv=3,
            scoring="neg_mean_squared_error",
            n_jobs=1
        )
        gs.fit(X, Y)
        tuned[name] = gs.best_estimator_

    return tuned

In [ ]:
def cross_fit_nuisances_fast(X, D, Y, learner, K=2):
    n = X.shape[0]
    m0_hat = np.zeros(n)
    m1_hat = np.zeros(n)
    e_hat = np.zeros(n)
    prop_model = LogisticRegression(max_iter=1000)
    kf = KFold(n_splits=K, shuffle=True, random_state=123)

    for train_idx, test_idx in kf.split(X):
        X_tr, X_te = X[train_idx], X[test_idx]
        D_tr = D[train_idx]
        Y_tr = Y[train_idx]

        prop = clone(prop_model)
        prop.fit(X_tr, D_tr)
        e_hat[test_idx] = prop.predict_proba(X_te)[:, 1]

        m1 = clone(learner)
        m0 = clone(learner)

        m1.fit(X_tr[D_tr == 1], Y_tr[D_tr == 1])
        m0.fit(X_tr[D_tr == 0], Y_tr[D_tr == 0])

        m1_hat[test_idx] = m1.predict(X_te)
        m0_hat[test_idx] = m0.predict(X_te)

    return m0_hat, m1_hat, e_hat

In [ ]:
def aipw_inference(Y, D, m0_hat, m1_hat, e_hat, tau_true):
    n = len(Y)
    #e_hat = np.clip(e_hat, 0.2, 0.8)
    
    psi = (
        D * (Y - m1_hat) / e_hat
        - (1 - D) * (Y - m0_hat) / (1 - e_hat)
        + m1_hat - m0_hat
    )

    tau_hat = psi.mean()
    var_hat = np.mean((psi - tau_hat) ** 2)
    se = np.sqrt(var_hat / n)

    ci_low = tau_hat - 1.96 * se
    ci_high = tau_hat + 1.96 * se
    covered = (ci_low <= tau_true) and (tau_true <= ci_high)

    return tau_hat, se, covered

In [ ]:
def run_simulation_fast(dgp_func, tuned_learners, n, seed):
    rng = np.random.default_rng(seed)
    X, D, Y, tau_true = dgp_func(n, rng)

    results = {}

    for name, learner in tuned_learners.items():
        m0_hat, m1_hat, e_hat = cross_fit_nuisances_fast(
            X, D, Y, learner, K=2
        )

        tau_hat, se, covered = aipw_inference(
            Y, D, m0_hat, m1_hat, e_hat, tau_true
        )

        results[name] = {
            "tau": tau_hat,
            "se": se,
            "covered": covered
        }

    return results

In [61]:
def monte_carlo(dgp_func, learners, n, sims=200):
    mc_results = Parallel(n_jobs=-1)(
        delayed(run_simulation_fast)(dgp_func, learners, n, s)
        for s in range(sims)
    )

    rows = []
    for sim in mc_results:
        for learner, res in sim.items():
            rows.append({
                "Learner": learner,
                "tau": res["tau"],
                "se": res["se"],
                "covered": res["covered"]
            })

    df = pd.DataFrame(rows)

    # tau_true = 1.0 in all your DGPs
    _, _, _, tau_true = dgp_func(n, np.random.default_rng(0))
    
    summary = df.groupby("Learner").agg(
        Mean=("tau", "mean"),
        Bias=("tau", lambda x: x.mean() - tau_true),
        Variance=("tau", "var"),
        RMSE=("tau", lambda x: np.sqrt(np.mean((x - tau_true) ** 2))),
        Coverage=("covered", "mean"),
        Mean_SE=("se", "mean")
    )

    return summary

In [62]:
learners_regime = {
    "OLS": LinearRegression(),
    "Lasso": Lasso(max_iter=5000),
    "ElasticNet": ElasticNet(max_iter=5000),
    "RF": RandomForestRegressor(
        n_estimators=300,
        random_state=123,
        n_jobs=1
    ),
    "GB": GradientBoostingRegressor(
        n_estimators=300,
        random_state=123
    )
}

In [ ]:
print("Regime 1")
tuned_learners_1 = tune_once(dgp1, learners_regime, n=100)
print(monte_carlo(dgp1, tuned_learners_1, n=100, sims=100))


print("Regime 2")
tune_learners_2 = tune_once(dgp2, learners_regime, n=100)
print(monte_carlo(dgp2, tune_learners_2, n=100, sims=100))

print("Regime 3")
tuned_learners_3 = tune_once(dgp3, learners_regime, n=100)
print(monte_carlo(dgp3, tuned_learners_3, n=100, sims=100))

Regime 1


                Mean      Bias  Variance      RMSE  Coverage   Mean_SE
Learner                                                               
ElasticNet  2.565978  0.065978  0.090769  0.306944      0.95  0.280061
GB          2.559192  0.059192  0.113111  0.339829      0.94  0.305783
Lasso       2.570524  0.070524  0.090493  0.307508      0.96  0.280352
OLS         2.564834  0.064834  0.095719  0.314588      0.96  0.281745
RF          2.581396  0.081396  0.142921  0.384859      0.96  0.357224
Regime 2
                Mean      Bias  Variance      RMSE  Coverage   Mean_SE
Learner                                                               
ElasticNet  0.985251 -0.014749  0.130928  0.360327      0.95  0.324685
GB          0.983497 -0.016503  0.162179  0.401035      0.94  0.342920
Lasso       0.955293 -0.044707  0.182266  0.427133      0.91  0.334530
OLS         0.935877 -0.064123  0.240606  0.492251      0.91  0.360659
RF          1.000468  0.000468  0.131708  0.361096      0.96  0.3523

In [ ]:
# =====================================================
# DGP Design Helpers
# =====================================================
def make_group_design(n, rng, n_groups):
    """
    One-hot encoding of group instances.
    Each observation belongs to exactly one group.
    """
    group_id = rng.integers(0, n_groups, size=n)
    X = np.zeros((n, n_groups))
    X[np.arange(n), group_id] = 1.0
    return X, group_id

def make_group_sparse_beta(n_groups, s_unique, rng):
    """
    s_unique = number of unique coefficient values
    group sparsity = n_groups >> s_unique
    """
    unique_vals = rng.uniform(0.5, 1.5, size=s_unique)
    beta = np.resize(unique_vals, n_groups)
    return beta

In [ ]:
# ONE HOT ENCODING BASED - DGPs
def dgp1(rng, n):
    n_groups = 6       # e.g. (Low/Mid/High) × (Male/Female)
    # no sparsity

    X, _ = make_group_design(n, rng, n_groups)

    beta0 = rng.uniform(0.5, 1.5, size=n_groups)
    beta1 = beta0 + 1.0

    eps0 = rng.normal(0, 1, size=n)
    eps1 = rng.normal(0, 1, size=n)

    Y0 = X @ beta0 + eps0
    Y1 = X @ beta1 + eps1

    D = rng.binomial(1, 0.5, size=n)
    Y = D * Y1 + (1 - D) * Y0

    return X, D, Y, 1.0

In [ ]:
def dgp2(rng, n):
    n_groups = int(0.1 * n)
    s_unique = 5       # strong group sparsity

    X, _ = make_group_design(n, rng, n_groups)

    beta0 = make_group_sparse_beta(n_groups, s_unique, rng)
    beta1 = beta0 + 1.0

    eps0 = rng.normal(0, 1, size=n)
    eps1 = rng.normal(0, 1, size=n)

    Y0 = X @ beta0 + eps0
    Y1 = X @ beta1 + eps1

    D = rng.binomial(1, 0.5, size=n)
    Y = D * Y1 + (1 - D) * Y0

    return X, D, Y, 1.0

In [ ]:
def dgp3_anomaly(rng, n):
    n_groups = int(2.5 * n)
    s_unique = n_groups  # no sparsity at all

    X, _ = make_group_design(n, rng, n_groups)

    beta0 = rng.uniform(0.5, 1.5, size=n_groups)
    beta1 = beta0 + 1.0

    eps0 = rng.normal(0, 1, size=n)
    eps1 = rng.normal(0, 1, size=n)

    Y0 = X @ beta0 + eps0
    Y1 = X @ beta1 + eps1

    D = rng.binomial(1, 0.5, size=n)
    Y = D * Y1 + (1 - D) * Y0

    return X, D, Y, 1.0

In [ ]:
# =====================================================
# OLD Monte Carlo Simulation [NON-PARALLEL]
# =====================================================
def monte_carlo(dgp, learners, n, sims, n_groups, beta_g, p_g):
    rows = []
    for s in range(sims):
        rng = np.random.default_rng(s)
        X, D, Y, tau_true = dgp(rng, n, n_groups, beta_g, p_g)
        # print(f"Simulation {s+1}/{sims} - Design Matrix Rank: {np.linalg.matrix_rank(X)}") # Print rank of design matrix for debugging
        # print(f"Matrix Shape: {X.shape}") # Print shape of design matrix for debugging
        for name, learner in learners.items():
            m0, m1, e = cross_fit_nuisances_fast(X, D, Y, learner)
            tau, se, cov, e_hat = aipw(Y, D, m0, m1, e, tau_true)
            rows.append({
                "Learner": name,
                "tau": tau,
                "se": se,
                "covered": cov,
                "e_mean": e_hat.mean(),
                "e_var": e_hat.var()
            })
    df = pd.DataFrame(rows)
    
    return df.groupby("Learner").agg(
        Mean=("tau", "mean"),
        Bias=("tau", lambda x: x.mean() - tau_true),
        Variance=("tau", "var"),
        Mean_SD_Err=("se", "mean"),
        RMSE=("tau", lambda x: np.sqrt(np.mean((x - tau_true)**2))),
        CIWidth=("se", lambda x: 2*1.96*np.mean(x)),
        Coverage=("covered", "mean"),
    )

In [ ]:
# =====================================================
# OLD Hyperparameter Tuning [NON-PARALLEL]
# =====================================================
def tune_once(dgp_func, learners, n, n_groups, beta_g, p_g, seed=0):
    rng = np.random.default_rng(seed)
    X, D, Y, _ = dgp_func(rng, n, n_groups, beta_g, p_g)
    tuned = {}
    for name, model in learners.items():
        if name == "Lasso":
            grid = LASSO_GRID
        elif name == "ElasticNet":
            grid = EN_GRID
        elif name == "RF":
            grid = RF_GRID
        elif name == "GB":
            grid = GB_GRID
        elif name == "CatBoost":
            grid = CATBOOST_GRID
        else:
            tuned[name] = clone(model)
            continue
        tuned[name] = tune_learner(clone(model), grid, X, Y)
    return tuned
